# Genome-wide Differential CNVs — Regions Associated with Short PFI

Identify **all genomic regions** — across every chromosome, for both gains and deletions — whose copy-number burden is enriched in short-PFI patients compared to long-PFI patients.

Author: Franziska Niemeyer

In [ ]:
DATA_DIR      = "../cnv_inference/infercnv_runs/combined_stroma_ref"

REGIONS_FILE   = DATA_DIR + "/HMM_CNV_predictions.HMMi6.leiden.hmm_mode-subclusters.Pnorm_0.5.pred_cnv_regions.dat"
GENES_FILE     = DATA_DIR + "/HMM_CNV_predictions.HMMi6.leiden.hmm_mode-subclusters.Pnorm_0.5.pred_cnv_genes.dat"
GROUPINGS_FILE = DATA_DIR + "/infercnv.17_HMM_predHMMi6.leiden.hmm_mode-subclusters.observation_groupings.txt"
ADATA_PATH     = "../../../quality_control/primary-cohort/adata.h5ad"

PFI_CAT_COL = "PFI"
SAMPLE_COL  = "patient"

PFI_ORDER   = ["short", "medium", "long"]
PFI_PALETTE = {'short': '#C7844A', 'medium': '#456EAE', 'long': '#538984'}

GENE_ID_COL     = "gene_ids"   # adata.var column with Ensembl IDs, or None
GENE_SYMBOL_COL = None         # adata.var column with symbols, or None

CHR            = "chr19"
CHR_LENGTH     = 58_617_616   # GRCh38
BIN_SIZE       = 100_000      # 100 kb bins  (reduce to 50_000 for higher resolution)

BIN_SIZE_OVERVIEW = 500_000   # 500 kb bins for overview plots
BIN_SIZE_ZOOM     = 100_000   # 100 kb bins for zoomed-in plots
SMOOTH            = 3         # smoothing window size (bins)

MAX_LOSS_STATE = 2
MIN_GAIN_STATE = 4

SAMPLE_THRESHOLD = 0.2   # fraction of subclusters per sample that must show gain
N_SHORT_REQUIRED = 3     # minimum short-PFI samples meeting the threshold
DIFF_THRESHOLD   = 0.3   # minimum mean(short) - mean(long) gain fraction

HIGHLIGHT_GENES = {
    "ENSG00000130303": "BST2", 
    "ENSG00000105173": "CCNE1",
    "ENSG00000105221": "AKT2",
    "ENSG00000118046": "STK11",
    "ENSG00000085872": "CHERP",
    "ENSG00000099308": "MATK",
    "ENSG00000105058": "RAD23A",
}

OUT_DIR = "../outputs/mcr_results"
import os; os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy  as np
import pandas as pd
import matplotlib.pyplot    as plt
import matplotlib.patches   as mpatches
import matplotlib.ticker    as mticker
import matplotlib.gridspec  as gridspec
from   matplotlib.colors    import to_rgb
import seaborn as sns
from   scipy.stats          import mannwhitneyu, norm
from   scipy.ndimage        import uniform_filter1d
from   itertools            import combinations
from   statsmodels.stats.multitest import multipletests
import anndata as ad

In [ ]:
plt.rcdefaults()
plt.rcParams.update({
    "font.family":        "sans-serif",
    "font.sans-serif":    ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size":        10,
    "axes.titlesize":   12,
    "axes.labelsize":   11,   # x/y axis labels
    "xtick.labelsize":  10,
    "ytick.labelsize":  10,
    "legend.fontsize":  10,
    "figure.titlesize": 14,
    "xtick.major.size":   4,
    "ytick.major.size":   4,
    "axes.titleweight":   "bold",
    "axes.linewidth":     0.8,
    "figure.dpi":         300,
})

In [ ]:
# ── Build per-sample fraction vector for one chromosome ───────────────────
def build_frac_matrix(chr_regions, sample_subs, samples,
                      chr_len, bin_size, cnv_type):
    n_bins   = chr_len // bin_size + 1
    all_subs = chr_regions["subcluster"].unique()
    sub_idx  = {s: i for i, s in enumerate(all_subs)}

    # Gains: start neutral (3) and take the maximum state seen
    # Losses: start neutral (3) and take the minimum state seen
    fill_val = 3
    reducer  = np.maximum if cnv_type == "gain" else np.minimum

    state_mat = np.full((n_bins, len(all_subs)), fill_val, dtype=np.float32)

    for _, row in chr_regions.iterrows():
        si = sub_idx.get(row["subcluster"])
        if si is None:
            continue
        b0 = int(row["start"] // bin_size)
        b1 = min(int(row["end"] // bin_size), n_bins - 1)
        state_mat[b0:b1+1, si] = reducer(state_mat[b0:b1+1, si], row["state"])

    compare = ((lambda m: m >= MIN_GAIN_STATE) if cnv_type == "gain"
               else (lambda m: m <= MAX_LOSS_STATE))

    frac = {}
    for s in samples:
        subs  = sample_subs[s]
        valid = [sub for sub in subs if sub in sub_idx]
        if not valid:
            frac[s] = np.zeros(n_bins)
            continue
        cols    = [sub_idx[sub] for sub in valid]
        frac[s] = compare(state_mat[:, cols]).mean(axis=1)
    return frac


# ── Merge contiguous True bins into (start, end) blocks ───────────────────
def merge_bins(mask, bin_size, chr_len):
    blocks = []
    in_block = False
    for i, m in enumerate(mask):
        if m and not in_block:
            blk_start = i * bin_size
            in_block  = True
        elif not m and in_block:
            blocks.append((blk_start, i * bin_size))
            in_block = False
    if in_block:
        blocks.append((blk_start, chr_len))
    return blocks


# ── Gene map from adata.var ────────────────────────────────────────────────
def build_gene_map(adata_path, gene_id_col, gene_symbol_col):
    adata = ad.read_h5ad(adata_path)
    var   = adata.var.copy()
    id_series  = (var[gene_id_col].astype(str)
                  if gene_id_col  and gene_id_col  in var.columns
                  else pd.Series(var.index.astype(str), index=var.index))
    sym_series = (var[gene_symbol_col].astype(str)
                  if gene_symbol_col and gene_symbol_col in var.columns
                  else pd.Series(var.index.astype(str), index=var.index))
    return dict(zip(id_series.values, sym_series.values))

### Load inferCNV files and sample metadata

In [ ]:
regions = pd.read_csv(REGIONS_FILE, sep="\t")
regions["subcluster"] = regions["cell_group_name"].str.split(".", n=1).str[-1]
regions["sample"]     = regions["subcluster"].str.extract(r"^(H\d+)")

CHR_LENGTHS = (
    regions.groupby("chr")["end"]
    .max()
    .to_dict()
)
CHROMS = [f"chr{i}" for i in range(1, 23) if f"chr{i}" in CHR_LENGTHS]

genes_df = pd.read_csv(GENES_FILE, sep="\t")
genes_df["subcluster"] = genes_df["cell_group_name"].str.split(".", n=1).str[-1]
genes_df["sample"]     = genes_df["subcluster"].str.extract(r"^(H\d+)")

groupings = pd.read_csv(GROUPINGS_FILE, sep=" ", quotechar='"', index_col=0)
groupings.index.name = "barcode"
groupings = groupings.reset_index().rename(
    columns={"Dendrogram Group": "subcluster",
             "Annotation Group": "annotation_group"})
groupings["sample"] = groupings["subcluster"].str.extract(r"^(H\d+)")
sample_subs = groupings.groupby("sample")["subcluster"].unique()

print(f"Total region rows : {len(regions):,}")
print(f"Total gene rows   : {len(genes_df):,}")
print(f"Total spots       : {len(groupings):,}")
print(f"Chromosomes       : {sorted(regions['chr'].unique())}")

adata   = ad.read_h5ad(ADATA_PATH)
obs     = adata.obs[[PFI_CAT_COL]].copy()
obs["sample"] = (adata.obs[SAMPLE_COL].astype(str)
                    if SAMPLE_COL and SAMPLE_COL in adata.obs.columns
                    else adata.obs.index.str.extract(r"-(\d+)$")[0])
pfi_map = obs.groupby("sample")[PFI_CAT_COL].first().str.lower().str.strip().to_dict()

samples        = sorted(sample_subs.index)
short_samples  = [s for s in samples if pfi_map.get(s) == "short"]
medium_samples = [s for s in samples if pfi_map.get(s) == "medium"]
long_samples   = [s for s in samples if pfi_map.get(s) == "long"]

print(f"\nShort  ({len(short_samples)}): {short_samples}")
print(f"Medium ({len(medium_samples)}): {medium_samples}")
print(f"Long   ({len(long_samples)}): {long_samples}")

gene_map = build_gene_map(ADATA_PATH, GENE_ID_COL, GENE_SYMBOL_COL)
print(f"\nGene map: {len(gene_map):,} entries from adata.var")

### Genome-wide scan at 1 Mb resolution

For every 1 Mb bin across all chromosomes and both CNV types (gain / loss) we compute:
- Gain/loss fraction per sample at that bin
- Differential score (mean short − mean long)

In [ ]:
print("Running genome-wide scan at 1 Mb resolution, short vs long...")
scan_rows = []

for cnv_type in ["gain", "loss"]:
    for chrom in CHROMS:
        if chrom not in CHR_LENGTHS:
            continue
        chr_r   = regions[regions["chr"] == chrom]
        chr_len = CHR_LENGTHS[chrom]
        n_bins  = chr_len // BIN_SIZE_OVERVIEW + 1

        if len(chr_r) == 0:
            for b in range(n_bins):
                scan_rows.append({
                    "chr": chrom, "bin": b, "cnv_type": cnv_type,
                    "start": b * BIN_SIZE_OVERVIEW,
                    "end"  : min((b+1) * BIN_SIZE_OVERVIEW, chr_len),
                    **{f"frac_{s}": 0.0 for s in samples},
                    "mean_short": 0.0, "mean_medium": 0.0, "mean_long": 0.0,
                    "diff": 0.0, "n_short_pass": 0, "flagged": False,
                })
            continue

        frac = build_frac_matrix(chr_r, sample_subs, samples,
                                  chr_len, BIN_SIZE_OVERVIEW, cnv_type)

        for b in range(n_bins):
            short_vals  = [frac[s][b] for s in short_samples]
            medium_vals = [frac[s][b] for s in medium_samples]
            long_vals   = [frac[s][b] for s in long_samples]

            diff         = float(np.mean(short_vals) - np.mean(long_vals))
            n_short_pass = sum(v >= SAMPLE_THRESHOLD for v in short_vals)

            row = {
                "chr"        : chrom,
                "bin"        : b,
                "cnv_type"   : cnv_type,
                "start"      : b * BIN_SIZE_OVERVIEW,
                "end"        : min((b+1) * BIN_SIZE_OVERVIEW, chr_len),
                "mean_short" : float(np.mean(short_vals)),
                "mean_medium": float(np.mean(medium_vals)),
                "mean_long"  : float(np.mean(long_vals)),
                "diff"       : diff,
                "n_short_pass": n_short_pass,
                "flagged"    : (diff          >= DIFF_THRESHOLD and
                                n_short_pass  >= N_SHORT_REQUIRED),
            }
            for s in samples:
                row[f"frac_{s}"] = float(frac[s][b])
            scan_rows.append(row)

scan_df = pd.DataFrame(scan_rows)

scan_df["significant"] = scan_df["flagged"]

n_sig_gain = scan_df[(scan_df["cnv_type"] == "gain") & scan_df["significant"]].shape[0]
n_sig_loss = scan_df[(scan_df["cnv_type"] == "loss") & scan_df["significant"]].shape[0]
print(f"Flagged bins — gain: {n_sig_gain}  |  loss: {n_sig_loss}")
print(f"Total bins scanned : {len(scan_df):,}")
print(f"Flagging criteria  : diff >= {DIFF_THRESHOLD}  AND  "
      f">= {N_SHORT_REQUIRED}/{len(short_samples)} short-PFI samples "
      f"with gain/loss fraction >= {SAMPLE_THRESHOLD}")

scan_df.to_csv(f"{OUT_DIR}/genome_scan_1Mb.tsv", sep="\t", index=False)
print(f"Saved: {OUT_DIR}/genome_scan_1Mb.tsv")

### Genome-wide overview

A linear genome-wide plot showing the differential score (mean short − mean long) for both gains (top, red) and losses (bottom, blue) across all chromosomes.
Significant bins are highlighted. Chromosomes alternate in shading for readability.


In [ ]:
# Assign a cumulative genomic offset to each chromosome for plotting
chrom_order = CHROMS
offsets = {}
cumpos  = 0
for c in chrom_order:
    offsets[c] = cumpos
    cumpos     += CHR_LENGTHS.get(c, 0) + 5_000_000  # 5Mb gap between chroms

fig, (ax_gain, ax_loss) = plt.subplots(2, 1, figsize=(10, 3.5),
                                        sharex=True, sharey=False)

for ax, cnv_type, label, base_color in [
    (ax_gain, "gain", "Gain", "#C0392B"),
    (ax_loss, "loss", "Loss", "#2980B9"),
]:
    sub = scan_df[scan_df["cnv_type"] == cnv_type].copy()

    for ci, chrom in enumerate(chrom_order):
        chr_sub = sub[sub["chr"] == chrom]
        if len(chr_sub) == 0:
            continue
        offset = offsets[chrom]

        x = (chr_sub["start"] + chr_sub["end"]) / 2 / 1e6 + offset / 1e6

        # Alternating chromosome background
        chr_end_mb = (offset + CHR_LENGTHS.get(chrom, 0)) / 1e6
        chr_start_mb = offset / 1e6
        ax.axvspan(chr_start_mb, chr_end_mb,
                   color="#F5F5F5" if ci % 2 == 0 else "#EBEBEB",
                   alpha=0.5, linewidth=0, zorder=0)

        # All bins: grey
        ax.bar(x, chr_sub["diff"], width=1.0,
               color="#CCCCCC", linewidth=0, zorder=1)

        # Significant bins: coloured
        sig_sub = chr_sub[chr_sub["significant"]]
        if len(sig_sub):
            x_sig = (sig_sub["start"] + sig_sub["end"]) / 2 / 1e6 + offset / 1e6
            ax.bar(x_sig, sig_sub["diff"], width=1.1,
                   color=base_color, linewidth=0, alpha=0.9, zorder=2)

    ax.axhline(DIFF_THRESHOLD, color="black", linewidth=0.8,
               linestyle="--", alpha=0.5, label=f"Diff threshold ({DIFF_THRESHOLD})")
    ax.axhline(0, color="black", linewidth=0.5, alpha=0.4)
    ax.set_ylabel(f"{label}\nshort − long", fontsize=9)
    ax.set_ylim(0, 1.05)
    ax.set_xticks([])  # Hide x-ticks since we have chromosome labels
    ax.set_xticklabels([])
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.legend(fontsize=8, loc="upper right")

    # Chromosome labels
    for chrom in chrom_order:
        mid_mb = (offsets[chrom] + CHR_LENGTHS.get(chrom, 0) / 2) / 1e6
        ax.text(mid_mb, -0.08, chrom.replace("chr", ""),
                ha="center", va="top", fontsize=7, color="#555555")

ax_loss.set_xlabel(f"Genomic region (chromosome)", fontsize=9, labelpad=15)
ax_gain.set_title("Differential copy-number variations — short-PFI enrichment",
                  fontsize=13, fontweight="bold")

sig_patch = mpatches.Patch(color="#3B3A3A", label=f"Flagged (difference ≥ {DIFF_THRESHOLD}, n short PFI patients ≥ {N_SHORT_REQUIRED})")
ns_patch  = mpatches.Patch(color="#CCCCCC", label="Not flagged")
fig.legend(handles=[sig_patch, ns_patch], loc="lower center",
           ncol=2, fontsize=9, bbox_to_anchor=(0.5, -0.08))

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/genome_scan_overview.pdf", bbox_inches="tight")
plt.savefig(f"{OUT_DIR}/genome_scan_overview.png", dpi=600, bbox_inches="tight")
plt.show()
print(f"Saved: {OUT_DIR}/genome_scan_overview.pdf")

### Candidate region table

Contiguous significant 1 Mb bins are merged into candidate regions.
Results are sorted by mean differential score (strongest association first).


In [ ]:
candidate_records = []

for cnv_type in ["gain", "loss"]:
    for chrom in CHROMS:
        sub = scan_df[
            (scan_df["chr"] == chrom) &
            (scan_df["cnv_type"] == cnv_type)
        ].sort_values("bin")

        if len(sub) == 0:
            continue

        sig_mask = sub["significant"].values
        chr_len  = CHR_LENGTHS.get(chrom, sub["end"].max())
        blocks   = merge_bins(sig_mask, BIN_SIZE_OVERVIEW, chr_len)

        for b_start, b_end in blocks:
            block_rows = sub[(sub["start"] >= b_start) & (sub["end"] <= b_end + 1)]
            if len(block_rows) == 0:
                continue

            short_cols = [f"frac_{s}" for s in short_samples]
            long_cols  = [f"frac_{s}" for s in long_samples]

            mean_short = block_rows[short_cols].values.mean()
            mean_long  = block_rows[long_cols].values.mean()
            best_diff  = block_rows["diff"].max()

            # Gene content
            block_genes = genes_df[
                (genes_df["chr"] == chrom) &
                (genes_df["start"] <= b_end) &
                (genes_df["end"]   >= b_start)
            ]["gene"].unique()
            symbols = [gene_map.get(g, g) for g in block_genes]
            named   = [s for s in symbols if not s.startswith("ENSG")]

            candidate_records.append({
                "chr"             : chrom,
                "start"           : b_start,
                "end"             : b_end,
                "size_Mb"         : round((b_end - b_start) / 1e6, 2),
                "cnv_type"        : cnv_type,
                "n_bins"          : len(block_rows),
                "mean_short"      : round(mean_short, 3),
                "mean_long"       : round(mean_long,  3),
                "diff"            : round(best_diff,  3),
                "n_inferCNV_genes": len(block_genes),
                "named_genes"     : "; ".join(named) if named else "—",
            })

candidates_df = pd.DataFrame(candidate_records)
if len(candidates_df):
    candidates_df = candidates_df.sort_values("diff", ascending=False)
else:
    candidates_df = pd.DataFrame(columns=[
        "chr","start","end","size_Mb","cnv_type","n_bins",
        "mean_short","mean_long","diff","n_inferCNV_genes","named_genes"
    ])

print(f"Candidate regions: {len(candidates_df)}")
print()
if len(candidates_df):
    print(candidates_df[[
        "chr","start","end","size_Mb","cnv_type",
        "mean_short","mean_long","diff","named_genes"
    ]].to_string(index=False))

    candidates_df.to_csv(f"{OUT_DIR}/candidate_regions.tsv", sep="\t", index=False)
    bed = candidates_df[["chr","start","end","cnv_type"]].copy()
    bed["name"]  = candidates_df["chr"] + "_" + candidates_df["cnv_type"]
    bed["score"] = (candidates_df["diff"] * 1000).clip(0, 1000).astype(int)
    bed.to_csv(f"{OUT_DIR}/candidate_regions.bed", sep="\t", index=False, header=False)
    print(f"\nSaved: {OUT_DIR}/candidate_regions.tsv")
    print(f"Saved: {OUT_DIR}/candidate_regions.bed")

### Per-chromosome profile plots for candidate chromosomes

For each chromosome with at least one candidate region we plot:
- **Top panel**: per-sample gain/loss fraction profile, one figure per PFI group
  (shades of the group colour, same style as the chr19 analysis)
- **Bottom panel**: differential score (short − long) with significant bins highlighted


In [ ]:
from matplotlib.colors import to_rgb

GOI = {"BST2": "ENSG00000130303",
       "COL13A1": "ENSG00000197467", "IFI27": "ENSG00000165949"}

def make_shades(hex_color, n, lightness_range=(0.45, 1.0)):
    base   = np.array(to_rgb(hex_color))
    white  = np.ones(3)
    alphas = np.linspace(lightness_range[1], lightness_range[0], n)
    return [tuple(a * base + (1 - a) * white) for a in alphas]

groups = {cat: [s for s in samples if pfi_map.get(s) == cat]
          for cat in PFI_ORDER}
sample_color = {}
for cat, grp in groups.items():
    shades = make_shades(PFI_PALETTE[cat], max(len(grp), 1))
    for s, c in zip(grp, shades):
        sample_color[s] = c

SMOOTH = 3

sig_chroms = sorted(candidates_df["chr"].unique()) if len(candidates_df) else []
print(f"Chromosomes with candidate regions: {sig_chroms}")

for chrom in sig_chroms:
    chr_len   = CHR_LENGTHS.get(chrom, 0)
    chr_cands = candidates_df[candidates_df["chr"] == chrom]

    for cnv_type in ["gain", "loss"]:
        type_cands = chr_cands[chr_cands["cnv_type"] == cnv_type]
        if len(type_cands) == 0:
            continue

        chr_r    = regions[regions["chr"] == chrom]
        frac     = build_frac_matrix(chr_r, sample_subs, samples,
                                      chr_len, BIN_SIZE_ZOOM, cnv_type)
        n_bins   = chr_len // BIN_SIZE_ZOOM + 1
        bin_mids = (np.arange(n_bins) + 0.5) * BIN_SIZE_ZOOM / 1e6

        scan_sub = scan_df[
            (scan_df["chr"] == chrom) &
            (scan_df["cnv_type"] == cnv_type)
        ].sort_values("bin")
        diff_1mb = scan_sub["diff"].values
        x_1mb    = ((scan_sub["start"].values + scan_sub["end"].values) / 2) / 1e6

        base_color = "#C0392B" if cnv_type == "gain" else "#2980B9"
        type_label = "Gain"   if cnv_type == "gain" else "Loss"

        # ── One figure per chromosome × CNV type ──────────────────────────
        n_profile_panels = len(PFI_ORDER)
        fig, axes = plt.subplots(
            n_profile_panels + 1, 1,
            figsize=(10, 2.5 * n_profile_panels),
            gridspec_kw={
                "height_ratios": [1] * n_profile_panels + [1],
                "hspace": 0.08,
            },
            sharex=True,
        )

        # ── One profile panel per PFI group ───────────────────────────────
        for i, cat in enumerate(PFI_ORDER):
            ax   = axes[i]
            grp  = groups[cat]

            for s in grp:
                gf = uniform_filter1d(frac[s], size=SMOOTH)
                ax.plot(bin_mids, gf, color=sample_color[s],
                        linewidth=1.6, alpha=0.9, label=s)

            ax.set_ylim(0, 1.05)
            ax.set_yticks([0, 0.25, 0.5, 0.75, 1])
            ax.set_ylabel("Fraction of\nsubclusters", fontsize=8)
            ax.legend(fontsize=7, loc="upper right", framealpha=0.85,
                      title=f"PFI {cat}", title_fontsize=7)
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)

            # Shade candidate regions
            for _, row in type_cands.iterrows():
                ax.axvspan(row["start"] / 1e6, row["end"] / 1e6,
                           color=base_color, alpha=0.08, zorder=0)

            # GOI markers
            chr_gene_rows = (genes_df[genes_df["chr"] == chrom]
                             .drop_duplicates("gene"))
            chr_genes_goi = chr_gene_rows[
                chr_gene_rows["gene"].isin(GOI.values())
            ]
            for _, grow in chr_genes_goi.iterrows():
                sym = gene_map.get(grow["gene"], "")
                if not sym or sym.startswith("ENSG"):
                    continue
                ax.axvline(grow["start"] / 1e6, color="gray",
                           linewidth=0.4, linestyle=":", alpha=0.4, zorder=0)

        axes[0].set_title(f"{chrom}  ·  {type_label}",
                          fontsize=11, fontweight="bold")

        # ── Differential score panel (bottommost) ─────────────────────────
        ax_diff     = axes[-1]
        smooth_diff = uniform_filter1d(diff_1mb, size=1)

        ax_diff.fill_between(x_1mb, smooth_diff,
                              where=smooth_diff >= 0,
                              color=PFI_PALETTE["short"], alpha=0.75,
                              linewidth=0, step="mid", label="Short > long")
        ax_diff.fill_between(x_1mb, smooth_diff,
                              where=smooth_diff < 0,
                              color=PFI_PALETTE["long"], alpha=0.75,
                              linewidth=0, step="mid", label="Long > short")
        ax_diff.axhline(DIFF_THRESHOLD, color="black", linewidth=0.8,
                         linestyle="--", alpha=0.5)
        ax_diff.axhline(0, color="black", linewidth=0.5, alpha=0.4)

        for _, row in type_cands.iterrows():
            ax_diff.axvspan(row["start"] / 1e6, row["end"] / 1e6,
                             color=base_color, alpha=0.15, zorder=0)

        # GOI labels on diff panel only (avoid clutter on profile panels)
        for _, grow in chr_genes_goi.iterrows():
            sym = gene_map.get(grow["gene"], "")
            if not sym or sym.startswith("ENSG"):
                continue
            pos_mb = grow["start"] / 1e6
            ax_diff.axvline(pos_mb, color="black", linewidth=0.8,
                             linestyle=":", alpha=0.7, zorder=0)
            ax_diff.text(pos_mb, ax_diff.get_ylim()[1] * 0.85, sym,
                          fontsize=8, rotation=90,
                          ha="right", va="top", color="black", fontweight='bold')

        ax_diff.set_ylabel("Short − long", fontsize=9)
        ax_diff.set_xlabel(f"{chrom} position (Mb)", fontsize=9)
        ax_diff.legend(fontsize=7, loc="upper right")
        ax_diff.spines["top"].set_visible(False)
        ax_diff.spines["right"].set_visible(False)

        plt.savefig(f"{OUT_DIR}/profile_{chrom}_{cnv_type}.pdf",
                    bbox_inches="tight")
        plt.savefig(f"{OUT_DIR}/profile_{chrom}_{cnv_type}.png",
                    dpi=600, bbox_inches="tight")
        plt.show()

### Combine long and medium PFI groups

In [ ]:
print("Running genome-wide scan at 1 Mb resolution, short vs (long + medium)...")
scan_rows = []

for cnv_type in ["gain", "loss"]:
    for chrom in CHROMS:
        if chrom not in CHR_LENGTHS:
            continue
        chr_r   = regions[regions["chr"] == chrom]
        chr_len = CHR_LENGTHS[chrom]
        n_bins  = chr_len // BIN_SIZE_OVERVIEW + 1

        if len(chr_r) == 0:
            for b in range(n_bins):
                scan_rows.append({
                    "chr": chrom, "bin": b, "cnv_type": cnv_type,
                    "start": b * BIN_SIZE_OVERVIEW,
                    "end"  : min((b+1) * BIN_SIZE_OVERVIEW, chr_len),
                    **{f"frac_{s}": 0.0 for s in samples},
                    "mean_short": 0.0, "mean_medium": 0.0, "mean_long": 0.0,
                    "diff": 0.0, "n_short_pass": 0, "flagged": False,
                })
            continue

        frac = build_frac_matrix(chr_r, sample_subs, samples,
                                  chr_len, BIN_SIZE_OVERVIEW, cnv_type)

        for b in range(n_bins):
            short_vals  = [frac[s][b] for s in short_samples]
            medium_vals = [frac[s][b] for s in medium_samples]
            long_vals   = [frac[s][b] for s in long_samples]

            diff = float(np.mean(short_vals) - np.mean(long_vals + medium_vals))
            n_short_pass = sum(v >= SAMPLE_THRESHOLD for v in short_vals)

            row = {
                "chr"        : chrom,
                "bin"        : b,
                "cnv_type"   : cnv_type,
                "start"      : b * BIN_SIZE_OVERVIEW,
                "end"        : min((b+1) * BIN_SIZE_OVERVIEW, chr_len),
                "mean_short" : float(np.mean(short_vals)),
                "mean_medium": float(np.mean(medium_vals)),
                "mean_long"  : float(np.mean(long_vals)),
                "diff"       : diff,
                "n_short_pass": n_short_pass,
                "flagged"    : (diff          >= DIFF_THRESHOLD and
                                n_short_pass  >= N_SHORT_REQUIRED),
            }
            for s in samples:
                row[f"frac_{s}"] = float(frac[s][b])
            scan_rows.append(row)

scan_df = pd.DataFrame(scan_rows)

scan_df["significant"] = scan_df["flagged"]

n_sig_gain = scan_df[(scan_df["cnv_type"] == "gain") & scan_df["significant"]].shape[0]
n_sig_loss = scan_df[(scan_df["cnv_type"] == "loss") & scan_df["significant"]].shape[0]
print(f"Flagged bins — gain: {n_sig_gain}  |  loss: {n_sig_loss}")
print(f"Total bins scanned : {len(scan_df):,}")
print(f"Flagging criteria  : diff >= {DIFF_THRESHOLD}  AND  "
      f">= {N_SHORT_REQUIRED}/{len(short_samples)} short-PFI samples "
      f"with gain/loss fraction >= {SAMPLE_THRESHOLD}")

scan_df.to_csv(f"{OUT_DIR}/genome_scan_1Mb.tsv", sep="\t", index=False)
print(f"Saved: {OUT_DIR}/genome_scan_1Mb.tsv")

In [ ]:
# Assign a cumulative genomic offset to each chromosome for plotting
chrom_order = CHROMS
offsets = {}
cumpos  = 0
for c in chrom_order:
    offsets[c] = cumpos
    cumpos     += CHR_LENGTHS.get(c, 0) + 5_000_000  # 5Mb gap between chroms

fig, (ax_gain, ax_loss) = plt.subplots(2, 1, figsize=(10, 4),
                                        sharex=True, sharey=False)

for ax, cnv_type, label, base_color in [
    (ax_gain, "gain", "Gain", "#C0392B"),
    (ax_loss, "loss", "Loss", "#2980B9"),
]:
    sub = scan_df[scan_df["cnv_type"] == cnv_type].copy()

    for ci, chrom in enumerate(chrom_order):
        chr_sub = sub[sub["chr"] == chrom]
        if len(chr_sub) == 0:
            continue
        offset = offsets[chrom]

        x = (chr_sub["start"] + chr_sub["end"]) / 2 / 1e6 + offset / 1e6

        # Alternating chromosome background
        chr_end_mb = (offset + CHR_LENGTHS.get(chrom, 0)) / 1e6
        chr_start_mb = offset / 1e6
        ax.axvspan(chr_start_mb, chr_end_mb,
                   color="#F5F5F5" if ci % 2 == 0 else "#EBEBEB",
                   alpha=0.5, linewidth=0, zorder=0)

        # All bins: grey
        ax.bar(x, chr_sub["diff"], width=1.0,
               color="#CCCCCC", linewidth=0, zorder=1)

        # Significant bins: coloured
        sig_sub = chr_sub[chr_sub["significant"]]
        if len(sig_sub):
            x_sig = (sig_sub["start"] + sig_sub["end"]) / 2 / 1e6 + offset / 1e6
            ax.bar(x_sig, sig_sub["diff"], width=1.1,
                   color=base_color, linewidth=0, alpha=0.9, zorder=2)

    ax.axhline(DIFF_THRESHOLD, color="black", linewidth=0.8,
               linestyle="--", alpha=0.5, label=f"Diff threshold ({DIFF_THRESHOLD})")
    ax.axhline(0, color="black", linewidth=0.5, alpha=0.4)
    ax.set_ylabel(f"{label}\nshort-(long+medium)")
    ax.set_ylim(0, 1.05)
    ax.set_xticks([])  # Hide x-ticks since we have chromosome labels
    ax.set_xticklabels([])
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.legend(loc="upper right")

    # Chromosome labels
    for chrom in chrom_order:
        mid_mb = (offsets[chrom] + CHR_LENGTHS.get(chrom, 0) / 2) / 1e6
        ax.text(mid_mb, -0.08, chrom.replace("chr", ""),
                ha="center", va="top", color="#555555")

ax_loss.set_xlabel(f"Genomic region (chromosome)", labelpad=18)
ax_gain.set_title("Differential copy-number variations — short-PFI enrichment",
                  fontweight="bold", pad=20)

sig_patch = mpatches.Patch(color="#3B3A3A", label=f"Flagged (difference ≥ {DIFF_THRESHOLD}, n short PFI patients ≥ {N_SHORT_REQUIRED})")
ns_patch  = mpatches.Patch(color="#CCCCCC", label="Not flagged")
fig.legend(handles=[sig_patch, ns_patch], loc="lower center",
           ncol=2, bbox_to_anchor=(0.5, -0.08))

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/genome_scan_overview_long+medium_vs_short.pdf", bbox_inches="tight")
plt.savefig(f"{OUT_DIR}/genome_scan_overview_long+medium_vs_short.png", dpi=600, bbox_inches="tight")
plt.show()
print(f"Saved: {OUT_DIR}/genome_scan_overview_long+medium_vs_short.pdf")

In [ ]:
candidate_records = []

for cnv_type in ["gain", "loss"]:
    for chrom in CHROMS:
        sub = scan_df[
            (scan_df["chr"] == chrom) &
            (scan_df["cnv_type"] == cnv_type)
        ].sort_values("bin")

        if len(sub) == 0:
            continue

        sig_mask = sub["significant"].values
        chr_len  = CHR_LENGTHS.get(chrom, sub["end"].max())
        blocks   = merge_bins(sig_mask, BIN_SIZE_OVERVIEW, chr_len)

        for b_start, b_end in blocks:
            block_rows = sub[(sub["start"] >= b_start) & (sub["end"] <= b_end + 1)]
            if len(block_rows) == 0:
                continue

            short_cols = [f"frac_{s}" for s in short_samples]
            medium_cols = [f"frac_{s}" for s in medium_samples]
            long_cols  = [f"frac_{s}" for s in long_samples]

            mean_short = block_rows[short_cols].values.mean()
            mean_medium = block_rows[medium_cols].values.mean()
            mean_long  = block_rows[long_cols].values.mean()
            best_diff  = block_rows["diff"].max()

            # Gene content
            block_genes = genes_df[
                (genes_df["chr"] == chrom) &
                (genes_df["start"] <= b_end) &
                (genes_df["end"]   >= b_start)
            ]["gene"].unique()
            symbols = [gene_map.get(g, g) for g in block_genes]
            named   = [s for s in symbols if not s.startswith("ENSG")]

            candidate_records.append({
                "chr"             : chrom,
                "start"           : b_start,
                "end"             : b_end,
                "size_Mb"         : round((b_end - b_start) / 1e6, 2),
                "cnv_type"        : cnv_type,
                "n_bins"          : len(block_rows),
                "mean_short"      : round(mean_short, 3),
                "mean_medium"      : round(mean_medium, 3),
                "mean_long"       : round(mean_long,  3),
                "diff"            : round(best_diff,  3),
                "n_inferCNV_genes": len(block_genes),
                "named_genes"     : "; ".join(named) if named else "—",
            })

candidates_df = pd.DataFrame(candidate_records)
if len(candidates_df):
    candidates_df = candidates_df.sort_values("diff", ascending=False)
else:
    candidates_df = pd.DataFrame(columns=[
        "chr","start","end","size_Mb","cnv_type","n_bins",
        "mean_short","mean_long","diff","n_inferCNV_genes","named_genes"
    ])

print(f"Candidate regions: {len(candidates_df)}")
print()
if len(candidates_df):
    print(candidates_df[[
        "chr","start","end","size_Mb","cnv_type",
        "mean_short","mean_medium","mean_long","diff","named_genes"
    ]].to_string(index=False))

    candidates_df.to_csv(f"{OUT_DIR}/candidate_regions_long+medium_vs_short.tsv", sep="\t", index=False)
    bed = candidates_df[["chr","start","end","cnv_type"]].copy()
    bed["name"]  = candidates_df["chr"] + "_" + candidates_df["cnv_type"]
    bed["score"] = (candidates_df["diff"] * 1000).clip(0, 1000).astype(int)
    bed.to_csv(f"{OUT_DIR}/candidate_regions_long+medium_vs_short.bed", sep="\t", index=False, header=False)
    print(f"\nSaved: {OUT_DIR}/candidate_regions_long+medium_vs_short.tsv")
    print(f"Saved: {OUT_DIR}/candidate_regions_long+medium_vs_short.bed")

In [ ]:
from matplotlib.colors import to_rgb

GOI = {"BST2": "ENSG00000130303",
       "COL13A1": "ENSG00000197467", "IFI27": "ENSG00000165949"}

def make_shades(hex_color, n, lightness_range=(0.45, 1.0)):
    base   = np.array(to_rgb(hex_color))
    white  = np.ones(3)
    alphas = np.linspace(lightness_range[1], lightness_range[0], n)
    return [tuple(a * base + (1 - a) * white) for a in alphas]

groups = {cat: [s for s in samples if pfi_map.get(s) == cat]
          for cat in PFI_ORDER}
sample_color = {}
for cat, grp in groups.items():
    shades = make_shades(PFI_PALETTE[cat], max(len(grp), 1))
    for s, c in zip(grp, shades):
        sample_color[s] = c

SMOOTH = 3

sig_chroms = sorted(candidates_df["chr"].unique()) if len(candidates_df) else []
print(f"Chromosomes with candidate regions: {sig_chroms}")

for chrom in sig_chroms:
    chr_len   = CHR_LENGTHS.get(chrom, 0)
    chr_cands = candidates_df[candidates_df["chr"] == chrom]

    for cnv_type in ["gain", "loss"]:
        type_cands = chr_cands[chr_cands["cnv_type"] == cnv_type]
        if len(type_cands) == 0:
            continue

        chr_r    = regions[regions["chr"] == chrom]
        frac     = build_frac_matrix(chr_r, sample_subs, samples,
                                      chr_len, BIN_SIZE_ZOOM, cnv_type)
        n_bins   = chr_len // BIN_SIZE_ZOOM + 1
        bin_mids = (np.arange(n_bins) + 0.5) * BIN_SIZE_ZOOM / 1e6

        scan_sub = scan_df[
            (scan_df["chr"] == chrom) &
            (scan_df["cnv_type"] == cnv_type)
        ].sort_values("bin")
        diff_1mb = scan_sub["diff"].values
        x_1mb    = ((scan_sub["start"].values + scan_sub["end"].values) / 2) / 1e6

        base_color = "#C0392B" if cnv_type == "gain" else "#2980B9"
        type_label = "Gain"   if cnv_type == "gain" else "Loss"

        # ── One figure per chromosome × CNV type ──────────────────────────
        n_profile_panels = len(PFI_ORDER)
        fig, axes = plt.subplots(
            n_profile_panels + 1, 1,
            figsize=(10, 2.5 * n_profile_panels),
            gridspec_kw={
                "height_ratios": [1] * n_profile_panels + [1],
                "hspace": 0.08,
            },
            sharex=True,
        )

        # ── One profile panel per PFI group ───────────────────────────────
        for i, cat in enumerate(PFI_ORDER):
            ax   = axes[i]
            grp  = groups[cat]

            for s in grp:
                gf = uniform_filter1d(frac[s], size=SMOOTH)
                ax.plot(bin_mids, gf, color=sample_color[s],
                        linewidth=1.6, alpha=0.9, label=s)

            ax.set_ylim(0, 1.05)
            ax.set_yticks([0, 0.25, 0.5, 0.75, 1])
            ax.set_ylabel("Fraction of\nsubclusters")
            ax.legend(loc="upper right", framealpha=0.85,
                      title=f"PFI {cat}")
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)

            # Shade candidate regions
            for _, row in type_cands.iterrows():
                ax.axvspan(row["start"] / 1e6, row["end"] / 1e6,
                           color=base_color, alpha=0.08, zorder=0)

            # GOI markers
            chr_gene_rows = (genes_df[genes_df["chr"] == chrom]
                             .drop_duplicates("gene"))
            chr_genes_goi = chr_gene_rows[
                chr_gene_rows["gene"].isin(GOI.values())
            ]
            for _, grow in chr_genes_goi.iterrows():
                sym = gene_map.get(grow["gene"], "")
                if not sym or sym.startswith("ENSG"):
                    continue
                ax.axvline(grow["start"] / 1e6, color="gray",
                           linewidth=0.4, linestyle=":", alpha=0.4, zorder=0)

        axes[0].set_title(f"Chromosome {chrom.removeprefix('chr')}  ·  {type_label}",
                          fontweight="bold")

        # ── Differential score panel (bottommost) ─────────────────────────
        ax_diff     = axes[-1]
        smooth_diff = uniform_filter1d(diff_1mb, size=1)

        ax_diff.fill_between(x_1mb, smooth_diff,
                              where=smooth_diff >= 0,
                              color=PFI_PALETTE["short"], alpha=0.75,
                              linewidth=0, step="mid", label="Short > (long + medium)")
        ax_diff.fill_between(x_1mb, smooth_diff,
                              where=smooth_diff < 0,
                              color=PFI_PALETTE["long"], alpha=0.75,
                              linewidth=0, step="mid", label="(Long + medium) > short")
        ax_diff.axhline(DIFF_THRESHOLD, color="black", linewidth=0.8,
                         linestyle="--", alpha=0.5)
        ax_diff.axhline(0, color="black", linewidth=0.5, alpha=0.4)

        for _, row in type_cands.iterrows():
            ax_diff.axvspan(row["start"] / 1e6, row["end"] / 1e6,
                             color=base_color, alpha=0.15, zorder=0)

        # GOI labels on diff panel only (avoid clutter on profile panels)
        for _, grow in chr_genes_goi.iterrows():
            sym = gene_map.get(grow["gene"], "")
            if not sym or sym.startswith("ENSG"):
                continue
            pos_mb = grow["start"] / 1e6
            ax_diff.axvline(pos_mb, color="black", linewidth=0.8,
                             linestyle=":", alpha=0.7, zorder=0)
            ax_diff.text(pos_mb, ax_diff.get_ylim()[1] * 0.85, sym,
                          rotation=90,
                          ha="right", va="top", color="black", fontweight='bold')

        ax_diff.set_ylabel("Short -\n(long + medium)")
        ax_diff.set_xlabel(f"Chromosome {chrom.removeprefix('chr')} position (Mb)")
        ax_diff.legend(loc="upper right")
        ax_diff.spines["top"].set_visible(False)
        ax_diff.spines["right"].set_visible(False)

        plt.savefig(f"{OUT_DIR}/profile_{chrom}_{cnv_type}_long+medium_vs_short.pdf",
                    bbox_inches="tight")
        plt.savefig(f"{OUT_DIR}/profile_{chrom}_{cnv_type}_long+medium_vs_short.png",
                    dpi=600, bbox_inches="tight")
        plt.show()